# RetinaNet — Fold 2 Model Tuning

У цьому notebook проводимо два експерименти для покращення RetinaNet: змінюємо learning rate та параметр Focal Loss.

Використовуємо те саме розбиття даних і налаштування оцінювання, що й для baseline. Базова модель Fold 2 навчена окремо.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
from zipfile import ZipFile

HW_ROOT = Path("/content/drive/MyDrive/AI_HW2")
DATA_ROOT = Path("/content/visdrone")
PREPROCESSED_ROOT = HW_ROOT / "preprocessed"

zip_path = HW_ROOT / "multiple-object-video-detection-hw-2-ucu-ai-2026.zip"
ready_file = DATA_ROOT / ".train_extracted"

assert zip_path.is_file(), "Dataset ZIP not found"

if not ready_file.exists():
    with ZipFile(zip_path) as z:
        train_files = [
            name for name in z.namelist()
            if name.startswith("train/") and not name.endswith("/")
        ]

        print(f"Extracting {len(train_files)} training files...")
        z.extractall(DATA_ROOT, members=train_files)

    ready_file.touch()
    print("Extraction completed")
else:
    print("Training data already extracted")

assert (DATA_ROOT / "train/images").is_dir()
assert (PREPROCESSED_ROOT / "train.json").is_file()
assert (PREPROCESSED_ROOT / "folds.json").is_file()

print("Dataset ready:", DATA_ROOT)


Extracting 66494 training files...
Extraction completed
Dataset ready: /content/visdrone


In [3]:
import json

PREPROCESSED_ROOT = HW_ROOT / "preprocessed"

with open(PREPROCESSED_ROOT / "train.json") as f:
    train_data = json.load(f)

with open(PREPROCESSED_ROOT / "folds.json") as f:
    fold_assignments = json.load(f)

assert len(train_data) == 24201
assert len(fold_assignments) == len(train_data)
assert set(fold_assignments) == {
    str(r["file_name"]) for r in train_data
}

print("Records:", len(train_data))
print("Folds:", set(fold_assignments.values()))


Records: 24201
Folds: {0, 1, 2}


In [4]:
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_ROOT = Path("/content/visdrone")

assert (DATA_ROOT / "train/images").is_dir(), (
    "Training images not found"
)

print("Device:", DEVICE)
print("Data root:", DATA_ROOT)


Device: cuda
Data root: /content/visdrone


In [5]:
%pip install -q pycocotools

In [6]:
import random
import re
from collections import defaultdict
import numpy as np

SEED = 42
FOLD = 2

N_SPLITS = 3
TRAIN_FRAME_STEP = 6
VAL_FRAME_STEP = 2

EPOCHS = 6
BATCH_SIZE = 2
LR = 1e-4

assert FOLD in range(N_SPLITS)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def sequence_name(rec):
    return Path(
        str(rec["file_name"]).replace("\\", "/")
    ).parent.name


def frame_sort_key(rec):
    name = Path(rec["file_name"]).stem
    match = re.search(r"(\d+)$", name)

    return (
        int(match.group(1)) if match else float("inf"),
        name
    )


def sample_by_sequence(records, step):
    grouped = defaultdict(list)

    for rec in records:
        grouped[sequence_name(rec)].append(rec)

    selected = []

    for sequence in sorted(grouped):
        frames = sorted(
            grouped[sequence],
            key=frame_sort_key
        )
        selected.extend(frames[::step])

    return selected


full_fold_train = [
    rec for rec in train_data
    if fold_assignments[rec["file_name"]] != FOLD
]

full_fold_val = [
    rec for rec in train_data
    if fold_assignments[rec["file_name"]] == FOLD
]

fold_train_data = sample_by_sequence(
    full_fold_train, TRAIN_FRAME_STEP
)

fold_val_data = sample_by_sequence(
    full_fold_val, VAL_FRAME_STEP
)

train_sequences = set(map(sequence_name, fold_train_data))
val_sequences = set(map(sequence_name, fold_val_data))

assert train_sequences.isdisjoint(val_sequences)
assert fold_train_data and fold_val_data

print(f"Fold {FOLD}")
print(f"Train: {len(full_fold_train)} -> {len(fold_train_data)}")
print(f"Validation: {len(full_fold_val)} -> {len(fold_val_data)}")
print("Train sequences:", len(train_sequences))
print("Val sequences:", len(val_sequences))


Fold 2
Train: 16162 -> 2713
Validation: 8039 -> 4024
Train sequences: 38
Val sequences: 18


In [7]:
import numpy as np
from PIL import Image
from torchvision import tv_tensors
from torchvision.transforms import v2 as T

class DetectionDataset(Dataset):
    def __init__(self, records, split, train=False):
        self.records = records
        self.split = split
        self.transforms = T.Compose(
            ([T.RandomHorizontalFlip(p=0.5)] if train else []) +
            [T.ToDtype(torch.float32, scale=True), T.ToPureTensor()]
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        path = DATA_ROOT / self.split / rec["file_name"]

        with Image.open(path) as img:
            image = np.array(img.convert("RGB"), copy=True)

        h, w = image.shape[:2]
        image = tv_tensors.Image(torch.from_numpy(image).permute(2, 0, 1))

        target = {
            "boxes": tv_tensors.BoundingBoxes(
                torch.tensor(rec["boxes"], dtype=torch.float32).reshape(-1, 4),
                format="XYXY", canvas_size=(h, w)
            ),
            "labels": torch.tensor(rec["labels"], dtype=torch.int64),
            "ignored_boxes": tv_tensors.BoundingBoxes(
                torch.tensor(rec["ignored_boxes"], dtype=torch.float32).reshape(-1, 4),
                format="XYXY", canvas_size=(h, w)
            )
        }

        return self.transforms(image, target)

def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = DetectionDataset(
    fold_train_data,
    split="train",
    train=True
)

val_dataset = DetectionDataset(
    fold_val_data,
    split="train",
    train=False
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))


Training images: 2713
Validation images: 4024


In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)


## Validation of the Fold 2 data

Перевіряємо правильність розбиття за `folds.json` та підготовлених даних. Для навчання використовуємо кожен 6-й кадр, для validation — кожен 2-й.

In [9]:
assert FOLD == 2
assert len(fold_val_data) == 4024, f"Unexpected Fold 2 validation size: {len(fold_val_data)}"
assert len(train_dataset) > 0 and len(val_dataset) > 0
sample_img, sample_target = train_dataset[0]
assert sample_img.dtype == torch.float32
assert set(sample_target['labels'].tolist()).issubset(set(range(1, 9)))
print(f"Fold {FOLD}: {len(train_dataset)} train / {len(val_dataset)} val images")
print("Device:", DEVICE)


Fold 2: 2713 train / 4024 val images
Device: cuda


## 2. Creating RetinaNet

Використовуємо RetinaNet із pretrained ResNet-50 та FPN, адаптовану до 8 класів датасету.

Зберігаємо обробку ignored regions, щоб модель не отримувала штраф за негативні anchors у цих областях. Це забезпечує однакові умови для обох експериментів.

In [10]:
from types import MethodType
from torchvision.models.detection import (
    retinanet_resnet50_fpn,
    RetinaNet_ResNet50_FPN_Weights,
    _utils as det_utils,
)
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.transform import resize_boxes
from torchvision.ops import box_iou, sigmoid_focal_loss
from tqdm.auto import tqdm

NUM_CLASSES = 9

def create_retinanet(focal_alpha=0.25):
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    model = retinanet_resnet50_fpn(
        weights=RetinaNet_ResNet50_FPN_Weights.DEFAULT
    )
    n_anchors = model.anchor_generator.num_anchors_per_location()[0]
    model.head.classification_head = RetinaNetClassificationHead(
        model.backbone.out_channels, n_anchors, NUM_CLASSES
    )


    if focal_alpha != 0.25:
        def focal_loss_with_alpha(self, targets, head_outputs, matched_idxs):
            per_image_losses = []
            for target, logits, matches in zip(
                targets, head_outputs['cls_logits'], matched_idxs
            ):
                fg = matches >= 0
                cls_targets = torch.zeros_like(logits)
                if fg.any():
                    cls_targets[fg, target['labels'][matches[fg]]] = 1.0
                valid = matches != det_utils.Matcher.BETWEEN_THRESHOLDS
                classification_loss = sigmoid_focal_loss(
                    logits[valid], cls_targets[valid],
                    alpha=focal_alpha, gamma=2.0, reduction='sum'
                )
                per_image_losses.append(
                    classification_loss / max(1, int(fg.sum().item()))
                )
            return sum(per_image_losses) / len(per_image_losses)
        model.head.classification_head.compute_loss = MethodType(
            focal_loss_with_alpha, model.head.classification_head
        )


    original_resize = model.transform.resize
    def resize_with_ignore(image, target):
        old_size = image.shape[-2:]
        image, target = original_resize(image, target)
        if target is not None and 'ignored_boxes' in target:
            target['ignored_boxes'] = resize_boxes(
                target['ignored_boxes'], old_size, image.shape[-2:]
            )
        return image, target
    model.transform.resize = resize_with_ignore

    def compute_loss_with_ignore(self, targets, head_outputs, anchors):
        matched_idxs = []
        for anchor, target in zip(anchors, targets):
            boxes = target['boxes']
            if len(boxes):
                matches = self.proposal_matcher(box_iou(boxes, anchor))
            else:
                matches = torch.full(
                    (len(anchor),), -1, dtype=torch.long, device=anchor.device
                )
            ignored = target.get('ignored_boxes')
            if ignored is not None and len(ignored):
                centers = (anchor[:, :2] + anchor[:, 2:]) / 2
                mask = torch.zeros(len(anchor), dtype=torch.bool, device=anchor.device)
                for box in ignored.unique(dim=0):
                    mask |= (
                        (centers[:, 0] >= box[0]) & (centers[:, 0] <= box[2]) &
                        (centers[:, 1] >= box[1]) & (centers[:, 1] <= box[3])
                    )
                matches[(matches < 0) & mask] = det_utils.Matcher.BETWEEN_THRESHOLDS
            matched_idxs.append(matches)
        return self.head.compute_loss(targets, head_outputs, anchors, matched_idxs)
    model.compute_loss = MethodType(compute_loss_with_ignore, model)
    return model.to(DEVICE)


## 3. Shared checkpoint and training functions

Визначаємо спільні функції для навчання обох експериментів. Після кожної епохи зберігаємо checkpoint на Google Drive, щоб за потреби можна було продовжити навчання.

In [11]:
import gc
import json
from pathlib import Path

EPOCHS = 6
BATCH_SIZE = 2

def prepare_run(model, experiment, lr, focal_alpha):
    output_dir = HW_ROOT / 'retinanet_tuning' / experiment / f'fold_{FOLD}'
    output_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = output_dir / 'fold_latest.pth'
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    start_epoch, losses = 0, []
    if ckpt_path.is_file():
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
        assert ckpt['fold'] == FOLD and ckpt['lr'] == lr
        assert ckpt['focal_alpha'] == focal_alpha
        assert ckpt['epochs'] == EPOCHS
        assert ckpt['train_frame_step'] == TRAIN_FRAME_STEP
        assert ckpt['val_frame_step'] == VAL_FRAME_STEP
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scaler.load_state_dict(ckpt['scaler'])
        start_epoch, losses = ckpt['epoch'], ckpt['train_losses']
        del ckpt
    print(f'{experiment} | fold {FOLD} | resume epoch {start_epoch}/{EPOCHS}')
    return optimizer, scaler, output_dir, ckpt_path, start_epoch, losses


def train_run(model, optimizer, scaler, ckpt_path, start_epoch, losses,
              experiment, lr, focal_alpha):
    assert DEVICE.type == 'cuda', 'Enable GPU for training'
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        totals = {'classification': 0.0, 'bbox_regression': 0.0}
        seen = 0
        for images, targets in tqdm(
            train_loader, desc=f'{experiment}: epoch {epoch+1}/{EPOCHS}', leave=False
        ):
            images = [image.to(DEVICE) for image in images]
            targets = [{key: value.to(DEVICE) for key, value in t.items()} for t in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())
            if not torch.isfinite(loss):
                raise RuntimeError(f'Non-finite loss at epoch {epoch+1}')
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            n = len(images)
            seen += n
            for k in totals:
                totals[k] += loss_dict[k].detach().item() * n
        avg = {k: v / seen for k, v in totals.items()}
        losses.append(sum(avg.values()))
        payload = dict(
            experiment=experiment, fold=FOLD, epoch=epoch+1, epochs=EPOCHS,
            lr=lr, focal_alpha=focal_alpha,
            train_frame_step=TRAIN_FRAME_STEP, val_frame_step=VAL_FRAME_STEP,
            model=model.state_dict(), optimizer=optimizer.state_dict(),
            scaler=scaler.state_dict(), train_losses=list(losses),
        )
        temp_path = ckpt_path.with_suffix('.tmp')
        torch.save(payload, temp_path)
        temp_path.replace(ckpt_path)
        assert ckpt_path.is_file()
        print(f"Epoch {epoch+1}: total={losses[-1]:.4f} | "
              f"classification={avg['classification']:.4f} | "
              f"bbox={avg['bbox_regression']:.4f}", flush=True)
    return losses


## 4. COCO evaluator

Використовуємо COCO evaluation для обчислення mAP, AP50, AP75 та AP для об'єктів різного розміру.

Predictions зберігаємо до обчислення метрик, щоб не втратити результати inference у разі переривання Colab.

In [12]:
import time
import json
import numpy as np

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm.auto import tqdm


CLASS_NAMES = {
    1: "person",
    2: "bicycle",
    3: "car",
    4: "van",
    5: "truck",
    6: "tricycle",
    7: "bus",
    8: "motor"
}


def xyxy_to_xywh(box):
    x1, y1, x2, y2 = [float(v) for v in box]
    return [
        x1, y1,
        max(0.0, x2 - x1),
        max(0.0, y2 - y1)
    ]


def build_coco_gt(records):
    images = []
    annotations = []
    annotation_id = 1

    for rec in records:
        image_id = int(rec["image_id"])

        images.append({
            "id": image_id,
            "file_name": rec["file_name"]
        })

        for box, label in zip(rec["boxes"], rec["labels"]):
            bbox = xyxy_to_xywh(box)

            if bbox[2] <= 0 or bbox[3] <= 0:
                continue

            annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": int(label),
                "bbox": bbox,
                "area": bbox[2] * bbox[3],
                "iscrowd": 0
            })
            annotation_id += 1

        for box in rec["ignored_boxes"]:
            bbox = xyxy_to_xywh(box)

            if bbox[2] <= 0 or bbox[3] <= 0:
                continue

            for label in CLASS_NAMES:
                annotations.append({
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": label,
                    "bbox": bbox,
                    "area": bbox[2] * bbox[3],
                    "iscrowd": 1
                })
                annotation_id += 1

    coco = COCO()
    coco.dataset = {
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id": i, "name": name}
            for i, name in CLASS_NAMES.items()
        ],
        "info": {},
        "licenses": []
    }
    coco.createIndex()

    return coco


In [13]:
import time
import io
import contextlib
from pycocotools.cocoeval import COCOeval

def save_validation_predictions(model, output_dir):
    prediction_path = output_dir / 'predictions_quick.json'
    timing_path = output_dir / 'inference_times.json'
    if prediction_path.is_file() and timing_path.is_file():
        print('Using saved predictions:', prediction_path)
        return

    assert DEVICE.type == 'cuda', 'Use GPU for validation inference'
    model.eval()
    preds, durations = [], []
    record_index = 0
    with torch.inference_mode():
        for images, _ in tqdm(val_loader, desc='Validation inference', leave=False):
            gpu_images = [img.to(DEVICE, non_blocking=True) for img in images]
            torch.cuda.synchronize()
            start = time.perf_counter()
            outputs = model(gpu_images)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            durations.append(elapsed / len(images))
            for out in outputs:
                rec = fold_val_data[record_index]
                record_index += 1
                for box, label, score in zip(
                    out['boxes'].cpu().tolist(),
                    out['labels'].cpu().tolist(),
                    out['scores'].cpu().tolist()
                ):
                    if int(label) not in CLASS_NAMES:
                        continue
                    bbox = xyxy_to_xywh(box)
                    if bbox[2] <= 0 or bbox[3] <= 0:
                        continue
                    preds.append(dict(
                        image_id=int(rec['image_id']), category_id=int(label),
                        bbox=bbox, score=float(score)
                    ))
    assert record_index == len(fold_val_data)
    prediction_path.write_text(json.dumps(preds))
    timing_path.write_text(json.dumps(durations))
    print('Saved inference:', len(preds), 'predictions to', prediction_path)


def score_saved_predictions(output_dir, experiment, lr, alpha):
    prediction_path = output_dir / 'predictions_quick.json'
    timing_path = output_dir / 'inference_times.json'
    assert prediction_path.is_file(), 'Run inference first'
    predictions = json.loads(prediction_path.read_text())
    durations = json.loads(timing_path.read_text())
    with contextlib.redirect_stdout(io.StringIO()):
        gt = build_coco_gt(fold_val_data)
        if predictions:
            dt = gt.loadRes(predictions)
            ev = COCOeval(gt, dt, iouType='bbox')
            ev.params.imgIds = [int(r['image_id']) for r in fold_val_data]
            ev.params.catIds = list(CLASS_NAMES)
            ev.params.maxDets = [1,10,100]
            ev.evaluate()
            ev.accumulate()
            ev.summarize()
            stats = [float(v) for v in ev.stats]
        else:
            stats = [0.0] * 12
    mean_secs = float(np.mean(durations)) if durations else float('nan')
    metrics = dict(
        mAP=stats[0], AP50=stats[1], AP75=stats[2],
        AP_small=stats[3], AP_medium=stats[4], AP_large=stats[5],
        inference_ms_per_image=mean_secs*1000,
        FPS_equivalent_batch1=(1/mean_secs if mean_secs>0 else 0.0),
        images_evaluated=len(fold_val_data), num_predictions=len(predictions)
    )
    (output_dir / 'metrics_quick.json').write_text(json.dumps(metrics, indent=2))
    config = dict(
        experiment=experiment, fold=FOLD, lr=lr, focal_alpha=alpha,
        epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED,
        train_frame_step=TRAIN_FRAME_STEP, val_frame_step=VAL_FRAME_STEP,
        validation_images=len(fold_val_data),
        evaluation_note='COCO-style category-specific crowd regions for ignored boxes',
        torchvision_version=__import__('torchvision').__version__,
    )
    (output_dir / 'config.json').write_text(json.dumps(config, indent=2))
    print('Evaluation:', json.dumps(metrics, indent=2))
    print('Saved to:', output_dir)


## 5. Experiment 1 — Learning Rate Tuning

Зменшуємо learning rate з 1e-4 до 5e-5, щоб перевірити його вплив на якість детекції.

Навчаємо нову модель з тими самими параметрами, що й baseline, змінюючи лише learning rate.

In [14]:
EXP1_NAME = 'lr_5e-5'
EXP1_LR = 5e-5
EXP1_ALPHA = 0.25

model = create_retinanet(focal_alpha=EXP1_ALPHA)
optimizer, scaler, OUTPUT_DIR, checkpoint_path, start_epoch, train_losses = prepare_run(
    model, EXP1_NAME, EXP1_LR, EXP1_ALPHA
)
assert OUTPUT_DIR == HW_ROOT / 'retinanet_tuning' / EXP1_NAME / 'fold_2'


Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_coco-eeacb38b.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_coco-eeacb38b.pth


100%|██████████| 130M/130M [00:02<00:00, 50.0MB/s]


lr_5e-5 | fold 2 | resume epoch 0/6


In [15]:
train_losses = train_run(
    model, optimizer, scaler, checkpoint_path, start_epoch,
    train_losses, EXP1_NAME, EXP1_LR, EXP1_ALPHA
)


lr_5e-5: epoch 1/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Epoch 1: total=0.6342 | classification=0.3499 | bbox=0.2843


lr_5e-5: epoch 2/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Epoch 2: total=0.4901 | classification=0.2467 | bbox=0.2435


lr_5e-5: epoch 3/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Epoch 3: total=0.4505 | classification=0.2259 | bbox=0.2246


... (2 outputs left out)

lr_5e-5: epoch 5/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Epoch 5: total=0.3521 | classification=0.1523 | bbox=0.1998


lr_5e-5: epoch 6/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Epoch 6: total=0.3746 | classification=0.1837 | bbox=0.1909


In [16]:
assert checkpoint_path.is_file()
assert len(train_losses) == EPOCHS
save_validation_predictions(model, OUTPUT_DIR)


Validation inference:   0%|          | 0/4024 [00:00<?, ?it/s]

Saved inference: 445216 predictions to /content/drive/MyDrive/AI_HW2/retinanet_tuning/lr_5e-5/fold_2/predictions_quick.json


In [17]:
score_saved_predictions(OUTPUT_DIR, EXP1_NAME, EXP1_LR, EXP1_ALPHA)


Evaluation: {
  "mAP": 0.1370203018761976,
  "AP50": 0.27331344445265054,
  "AP75": 0.12482035546150554,
  "AP_small": 0.03946305954623576,
  "AP_medium": 0.1357402972204629,
  "AP_large": 0.34314417720804374,
  "inference_ms_per_image": 197.29076317469773,
  "FPS_equivalent_batch1": 5.068661015389334,
  "images_evaluated": 4024,
  "num_predictions": 445216
}
Saved to: /content/drive/MyDrive/AI_HW2/retinanet_tuning/lr_5e-5/fold_2


## 6. Experiment 2 — Focal Loss Modification

Змінюємо параметр alpha у Focal Loss з 0.25 до 0.5, щоб перевірити вплив балансу позитивних і негативних прикладів на якість детекції.

Learning rate залишаємо як у baseline (1e-4), gamma = 2 та інші налаштування не змінюємо.

In [18]:
import gc
for name in ['model','optimizer','scaler']:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

EXP2_NAME = 'focal_alpha_05'
EXP2_LR = 1e-4
EXP2_ALPHA = 0.5

model = create_retinanet(focal_alpha=EXP2_ALPHA)
optimizer, scaler, OUTPUT_DIR, checkpoint_path, start_epoch, train_losses = prepare_run(
    model, EXP2_NAME, EXP2_LR, EXP2_ALPHA
)
assert OUTPUT_DIR == HW_ROOT / 'retinanet_tuning' / EXP2_NAME / 'fold_2'


focal_alpha_05 | fold 2 | resume epoch 0/6


In [19]:
train_losses = train_run(
    model, optimizer, scaler, checkpoint_path, start_epoch,
    train_losses, EXP2_NAME, EXP2_LR, EXP2_ALPHA
)


focal_alpha_05: epoch 1/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Epoch 1: total=0.9620 | classification=0.6504 | bbox=0.3117


focal_alpha_05: epoch 2/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Epoch 2: total=0.6372 | classification=0.3775 | bbox=0.2596


focal_alpha_05: epoch 3/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Epoch 3: total=0.5589 | classification=0.3184 | bbox=0.2405


... (2 outputs left out)

focal_alpha_05: epoch 5/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Epoch 5: total=0.7407 | classification=0.4713 | bbox=0.2694


focal_alpha_05: epoch 6/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Epoch 6: total=0.8353 | classification=0.5414 | bbox=0.2940


In [20]:
assert checkpoint_path.is_file()
assert len(train_losses) == EPOCHS
save_validation_predictions(model, OUTPUT_DIR)


Validation inference:   0%|          | 0/4024 [00:00<?, ?it/s]

Saved inference: 570622 predictions to /content/drive/MyDrive/AI_HW2/retinanet_tuning/focal_alpha_05/fold_2/predictions_quick.json


In [21]:
score_saved_predictions(OUTPUT_DIR, EXP2_NAME, EXP2_LR, EXP2_ALPHA)


Evaluation: {
  "mAP": 0.10457644614638227,
  "AP50": 0.22562484382815157,
  "AP75": 0.08664331110400149,
  "AP_small": 0.03744009608662316,
  "AP_medium": 0.11552652140689953,
  "AP_large": 0.24161572857980854,
  "inference_ms_per_image": 196.28031200919924,
  "FPS_equivalent_batch1": 5.0947544853766695,
  "images_evaluated": 4024,
  "num_predictions": 570622
}
Saved to: /content/drive/MyDrive/AI_HW2/retinanet_tuning/focal_alpha_05/fold_2


## 7. Results and file verification

Порівнюємо метрики обох експериментів та перевіряємо збереження checkpoints, predictions і конфігурацій на Google Drive.

In [22]:
import pandas as pd
rows = []
for experiment in ('lr_5e-5', 'focal_alpha_05'):
    folder = HW_ROOT / 'retinanet_tuning' / experiment / f'fold_{FOLD}'
    files = {name: (folder / name).is_file() for name in [
        'fold_latest.pth','predictions_quick.json','inference_times.json',
        'metrics_quick.json','config.json'
    ]}
    print(experiment, files)
    if files['metrics_quick.json']:
        metrics = json.loads((folder / 'metrics_quick.json').read_text())
        rows.append(dict(experiment=experiment, **metrics))
if rows:
    display(pd.DataFrame(rows).round(4))


lr_5e-5 {'fold_latest.pth': True, 'predictions_quick.json': True, 'inference_times.json': True, 'metrics_quick.json': True, 'config.json': True}
focal_alpha_05 {'fold_latest.pth': True, 'predictions_quick.json': True, 'inference_times.json': True, 'metrics_quick.json': True, 'config.json': True}


,experiment,mAP,AP50,AP75,AP_small,AP_medium,AP_large,inference_ms_per_image,FPS_equivalent_batch1,images_evaluated,num_predictions
0,lr_5e-5,0.1370,0.2733,0.1248,0.0395,0.1357,0.3431,197.2908,5.0687,4024,445216
1,focal_alpha_05,0.1046,0.2256,0.0866,0.0374,0.1155,0.2416,196.2803,5.0948,4024,570622


### Висновок

Обидва експерименти покращили результати RetinaNet порівняно з baseline Fold 2 (mAP = 0.0576).

Найкращий результат отримано після зменшення learning rate до 5e-5: mAP зріс до 0.1370. Зміна Focal Loss alpha до 0.5 також покращила якість детекції (mAP = 0.1046), але результат виявився нижчим, ніж у першому експерименті.

Остаточні висновки щодо ефективності змін зробимо після порівняння всіх трьох folds.